# Tablero de Calidad de Datos — Reservas Contables DWH (solo lectura)

Este notebook es **solo de lectura**: ejecuta únicamente `SELECT` contra el DWH (nunca
`CREATE`/`INSERT`/`DELETE`). Es la versión con historia — y con más reglas — del control
`control_completitud_cuentas.sql`, que revisa un solo mes y una sola regla.

Qué hace:

1. Se conecta al DWH (SQL Server).
2. **Detecta automáticamente** los últimos `N_PERIODOS` periodos contables cargados — sin fechas quemadas.
3. Calcula los indicadores de calidad con **una sola pasada** por cada tabla fuente:
   - **Completitud** — ¿`VALOR_RESERVA_CONTABLE` viene nulo? (la regla original del control).
   - **Validez** — ¿la fila está bien formada? (`RAMO_PROD` presente en cedidas, periodo `AAAAMM` válido, libro no vacío).
   - **Variación (Estabilidad)** — ¿el valor o el volumen se movió más de lo normal frente a su propio histórico?
   - **Disponibilidad de carga** — ¿alguna combinación cuenta/libro se quedó sin filas ese mes? (huecos).
   - **Frescura** — ¿alguna de las 4 tablas fuente se quedó rezagada frente a las demás?
   - **Valores en cero** (informativo) — ¿cuántas filas traen el valor exactamente en 0?
4. Genera un **tablero HTML autocontenido** (`dashboard_completitud.html`) y un **CSV** con el
   cubo de indicadores en `reports/`.

**¿Cuándo correrlo?** Los días **8 o 9** de cada mes, antes del cierre oficial (día 10) —
es un control de alerta temprana: si algo falla, todavía hay tiempo de avisar al dueño de la tabla.

**Semáforo:** 🟢 COMPLETO = la cuenta está lista para el cierre · 🔴 ALERTA = avisar al dueño de la tabla.

**Cuentas que NO se revisan aquí** (tienen sus propios controles): Incurrido, Salvamentos,
Recobros, IVA AG (se excluye con el filtro `Libro <> 'AG'`) + 2 cuentas por confirmar con Andrey.

In [1]:
import json
import getpass
import webbrowser
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## Conexión al DWH

Las credenciales se cargan desde `credenciales_local.py` (archivo en `.gitignore`, vive solo en
este computador). Si el archivo no existe, se piden por consola sin quedar guardadas en ninguna parte.

In [2]:
# Credenciales fuera del repo: se cargan desde notebooks/credenciales_local.py,
# que está en .gitignore (así no se exponen y tampoco hay que digitarlas cada vez).
try:
    from credenciales_local import (
        DEFAULT_HOST_DWH, DEFAULT_PORT_DWH, DEFAULT_DATABASE_DWH,
        DEFAULT_USER_DWH, DEFAULT_PASSWORD_DWH,
    )
    print('Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).')
except ImportError:
    print('No se encontró credenciales_local.py — ingresa las credenciales:')
    DEFAULT_HOST_DWH = input('Host DWH: ')
    DEFAULT_PORT_DWH = input('Puerto [1433]: ') or '1433'
    DEFAULT_DATABASE_DWH = input('Base de datos: ')
    DEFAULT_USER_DWH = input('Usuario: ')
    DEFAULT_PASSWORD_DWH = getpass.getpass('Contraseña: ')

Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).


In [3]:
url = URL.create(
    'mssql+pyodbc',
    username=DEFAULT_USER_DWH,
    password=DEFAULT_PASSWORD_DWH,
    host=DEFAULT_HOST_DWH,
    port=int(DEFAULT_PORT_DWH),
    database=DEFAULT_DATABASE_DWH,
    query={'driver': 'SQL Server'},  # driver ODBC disponible en esta máquina
)
engine = create_engine(url)


def run_query(sql):
    '''Ejecuta un SELECT (solo lectura) y devuelve un DataFrame.'''
    return pd.read_sql(sql, engine)


run_query('SELECT 1 AS conexion_ok')

,conexion_ok
0,1


## 0) Fuentes de datos: ¿sobre qué se hace el DQ?

Todo el data quality de este notebook se calcula sobre las **4 tablas de interfaz de reservas**
del DWH (esquema `Liberty.RESERVAS`) — las mismas 4 fuentes, cuentas y filtros del control canónico:

| Fuente | Tabla | Negocio |
|---|---|---|
| `DIRECTA_IAXIS` | `DIRECTA_RESERVA_INTERFAZ` | Negocio directo (origen IAXIS) |
| `CEDIDAS_AS400` | `CEDIDAS_RESERVA_INTERFAZ` | Reaseguro cedido (origen AS400) |
| `CEDIDAS_IAXIS` | `CEDIDAS_RESERVA_INTERFAZ_IAXIS` | Reaseguro cedido (origen IAXIS) |
| `CEDIDAS_TERREMOTO` | `CEDIDAS_TERREMOTO_RESERVA_INTERFAZ` | Reserva de terremoto (filtro `FUENTE_INTERFAZ = 'TERR'`) |

> `CEDIDAS_AS400` y `CEDIDAS_TERREMOTO` **no tienen índice por periodo**: cualquier consulta las
> recorre completas y tarda varios minutos. Por eso este notebook hace **una sola pasada** por
> tabla y saca todos los conteos de una vez (antes eran dos pasadas).

In [4]:
# Este diccionario se reutiliza en la sección 0 (muestra), en el diccionario del
# tablero HTML (drill-down) y en la regla de Frescura (sección 2.4).
FUENTES = {
    'DIRECTA_IAXIS':     {'tabla': 'Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ',
                          'desc': 'negocio directo, origen IAXIS'},
    'CEDIDAS_AS400':     {'tabla': 'Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ',
                          'desc': 'reaseguro cedido, origen AS400'},
    'CEDIDAS_IAXIS':     {'tabla': 'Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS',
                          'desc': 'reaseguro cedido, origen IAXIS'},
    'CEDIDAS_TERREMOTO': {'tabla': 'Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ',
                          'desc': 'reserva de terremoto, interfaz TERR'},
}

# Muestra de 10 filas de la fuente rápida, para entender la base antes del DQ.
df_muestra = run_query('SELECT TOP 10 * FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ')
df_muestra.head()

,ramo_prod,poliza,certificado,anexo,smovrec,documento,amparo_orig,subamparo_orig,tipo_identi_aseg,nro_identi_aseg,...,CUENTA_COMISION,AUXILIAR_COMISION,TERCERO_COMISION,CUENTA_COMISION_WEB,AUXILIAR_COMISION_WEB,TERCERO_COMISION_WEB,BUSINESS_AREA,CHANNEL,RI_CODE,PROFIT_CENTER
0,01,898,0,20100700,None,0,9999,0,N,9001903859,...,None,None,None,None,None,None,None,None,None,None
1,01,898,0,20100700,None,0,9999,0,N,9001903859,...,None,None,None,None,None,None,None,None,None,None
2,01,898,0,20101000,None,1,9999,0,N,9001903859,...,None,None,None,None,None,None,None,None,None,None
3,01,898,0,20101000,None,1,9999,0,N,9001903859,...,None,None,None,None,None,None,None,None,None,None
4,01,898,0,20101200,None,3,9999,0,N,9001903859,...,None,None,None,None,None,None,None,None,None,None


In [5]:
df_muestra.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 61 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   ramo_prod                                10 non-null     str    
 1   poliza                                   10 non-null     int64  
 2   certificado                              10 non-null     int64  
 3   anexo                                    10 non-null     int64  
 4   smovrec                                  0 non-null      object 
 5   documento                                10 non-null     int64  
 6   amparo_orig                              10 non-null     str    
 7   subamparo_orig                           10 non-null     str    
 8   tipo_identi_aseg                         10 non-null     str    
 9   nro_identi_aseg                          10 non-null     str    
 10  periodo_contable_analisis                10 non-null     int64  

### Diccionario de campos del control

Documenta cada columna que participa en las reglas de este tablero: qué significa y en qué
indicador entra. Las definiciones se reutilizan en el tablero HTML (drill-down al hacer clic
en un anillo del resumen).

In [6]:
DICCIONARIO_CAMPOS = {
    'CUENTA': 'Cuenta contable de la reserva (grupos 41xxxx y 51xxxx). El control solo revisa '
              'las cuentas acordadas con el equipo. Participa en: todas las reglas (es parte del grano).',
    'LIBRO': "Libro contable. 'AG' se excluye en todas las fuentes (el IVA AG tiene control propio). "
             'Participa en: Validez (libro no vacío) y en el grano de todas las reglas.',
    'periodo_contable_analisis': 'Periodo contable AAAAMM de la fila (ej. 202606 = junio de 2026). '
                                 'Participa en: Validez (formato AAAAMM) y en el grano de todas las reglas.',
    'VALOR_RESERVA_CONTABLE': 'Valor de la reserva contable — el campo que vigila este control. '
                              'Participa en: Completitud (nulos), Variación (suma por mes) y Valores en cero.',
    'RAMO_PROD': 'Ramo del producto. En las fuentes cedidas el control canónico descarta las filas con '
                 'ramo nulo; aquí además las contamos como regla de Validez.',
    'FUENTE_INTERFAZ': "Solo en CEDIDAS_TERREMOTO: identifica la interfaz de origen; el control filtra 'TERR'.",
}
pd.DataFrame([{'campo': k, 'descripcion': v} for k, v in DICCIONARIO_CAMPOS.items()])

,campo,descripcion
0,CUENTA,Cuenta contable de la reserva (grupos 41xxxx y...
1,LIBRO,Libro contable. 'AG' se excluye en todas las f...
2,periodo_contable_analisis,Periodo contable AAAAMM de la fila (ej. 202606...
3,VALOR_RESERVA_CONTABLE,Valor de la reserva contable — el campo que vi...
4,RAMO_PROD,Ramo del producto. En las fuentes cedidas el c...
5,FUENTE_INTERFAZ,Solo en CEDIDAS_TERREMOTO: identifica la inter...


## 1) Detección automática de periodos (sin fechas quemadas)

En vez de escribir el periodo a mano, le preguntamos a las propias tablas cuál es el **último
periodo contable cargado** y desde ahí contamos `N_PERIODOS` meses hacia atrás. Así el tablero
siempre muestra la última ventana disponible sin editar nada. `PERIODOS_MANUAL` permite forzar
periodos puntuales si hace falta.

> Solo se consulta el máximo en `DIRECTA` y `CEDIDAS_IAXIS`, que responden en segundos.
> Las otras dos tablas tardan varios minutos incluso para un simple `MAX` (no tienen índice
> por periodo) y comparten el mismo calendario de carga.

In [7]:
N_PERIODOS = 12        # cuántos meses hacia atrás incluir en el tablero
PERIODOS_MANUAL = None  # ej. [202601, 202602] para forzar periodos puntuales

sql_ultimo_periodo = '''
SELECT MAX(p) AS ultimo_periodo
FROM (
    SELECT MAX(periodo_contable_analisis) AS p FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ
    UNION ALL
    SELECT MAX(periodo_contable_analisis)      FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS
) t
'''
ULTIMO_PERIODO = int(run_query(sql_ultimo_periodo).iloc[0, 0])


def periodos_hacia_atras(periodo_final, n=12):
    '''Lista de n periodos AAAAMM terminando en periodo_final (incluido).'''
    anio, mes = divmod(periodo_final, 100)
    periodos = []
    for _ in range(n):
        periodos.append(anio * 100 + mes)
        mes -= 1
        if mes == 0:
            anio, mes = anio - 1, 12
    return sorted(periodos)


if PERIODOS_MANUAL:
    PERIODOS = sorted(int(p) for p in PERIODOS_MANUAL)
    ULTIMO_PERIODO = PERIODOS[-1]
else:
    PERIODOS = periodos_hacia_atras(ULTIMO_PERIODO, N_PERIODOS)

lista_periodos = ', '.join(str(int(p)) for p in PERIODOS)

# ¿El último periodo cargado es "en curso" (mes calendario actual) o un mes ya cerrado?
hoy = datetime.now()
periodo_actual_calendario = int(hoy.strftime('%Y%m'))
PERIODO_EN_CURSO = ULTIMO_PERIODO if ULTIMO_PERIODO == periodo_actual_calendario else None

# Chequeo de frescura global: el DWH debería tener, como mínimo, el mes calendario anterior.
anio_ant, mes_ant = (hoy.year - 1, 12) if hoy.month == 1 else (hoy.year, hoy.month - 1)
PERIODO_ESPERADO = anio_ant * 100 + mes_ant

print(f'Último periodo cargado en DWH: {ULTIMO_PERIODO}')
print(f'Ventana de análisis ({len(PERIODOS)} meses): {PERIODOS[0]} → {PERIODOS[-1]}')
if PERIODO_EN_CURSO:
    print(f'El periodo {PERIODO_EN_CURSO} es el mes en curso: sus cifras son PARCIALES.')
elif ULTIMO_PERIODO < PERIODO_ESPERADO:
    print(f'⚠️  ATENCIÓN: se esperaba encontrar al menos el periodo {PERIODO_ESPERADO} '
          f'y el último cargado es {ULTIMO_PERIODO} — el DWH parece rezagado.')

Último periodo cargado en DWH: 202606
Ventana de análisis (12 meses): 202507 → 202606


## 2) Cálculo de los indicadores — una sola pasada por cada tabla

Antes este notebook recorría cada tabla **dos veces** (una para contar nulos y otra para sumar el
valor). Ahora una sola consulta trae todo, por combinación **fuente + cuenta + libro + mes**:

- `con_valor` / `sin_valor` → **Completitud** (idéntico al control canónico `control_completitud_cuentas.sql`).
- `ceros` → filas con el valor presente pero exactamente en **0** → métrica **Valores en cero**.
- `ramo_nulo` → filas con `RAMO_PROD` vacío → **Validez**. En las cedidas, el control canónico las
  descarta en silencio con `RAMO_PROD IS NOT NULL`; aquí quitamos ese filtro del `WHERE` para poder
  **contarlas**, pero los conteos de completitud siguen condicionados a ramo presente — así las
  cifras canónicas no cambian.
- `valor_total` → suma del valor → alimenta **Variación** sin una segunda pasada.

> ⏳ **Paciencia**: esta celda puede tardar **varios minutos** (dos tablas no tienen índice por
> periodo), pero es la única celda lenta del notebook.

In [8]:
# Los periodos son enteros calculados por nosotros (no texto del usuario),
# por eso es seguro insertarlos directamente en el SQL.
query_indicadores = f'''
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    total_registros,
    con_valor,
    sin_valor,
    ceros,
    ramo_nulo,
    valor_total
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        COUNT(*)        AS total_registros,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END) AS con_valor,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END) AS sin_valor,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE = 0         THEN 1 ELSE 0 END) AS ceros,
        SUM(CASE WHEN a.ramo_prod IS NULL                  THEN 1 ELSE 0 END) AS ramo_nulo,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))       AS valor_total
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400.
       Sin RAMO_PROD IS NOT NULL en el WHERE (para poder contar esas filas);
       los conteos canónicos van condicionados dentro de cada SUM(CASE ...). */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE = 0         THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL
                 THEN CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)) ELSE 0 END)
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS (misma estructura que AS400) */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL AND a.VALOR_RESERVA_CONTABLE = 0         THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.RAMO_PROD IS NOT NULL
                 THEN CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)) ELSE 0 END)
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO (RAMO_PROD no se valida en esta fuente) */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE = 0         THEN 1 ELSE 0 END),
        0,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
'''

df = run_query(query_indicadores)
for c in ['total_registros', 'con_valor', 'sin_valor', 'ceros', 'ramo_nulo']:
    df[c] = df[c].astype('int64')
df['valor_total'] = df['valor_total'].astype(float)
df['libro'] = df['libro'].astype(str).str.strip()
df['serie'] = df['fuente'] + ' · ' + df['cuenta'].astype(str) + ' · ' + df['libro']

# Universo canónico de completitud/variación: combinaciones con al menos una fila "válida".
# (En cedidas, una combinación que SOLO tenga filas con ramo nulo entra únicamente a Validez.)
df_control = df[df['total_registros'] > 0].copy()
df_control['pct_completitud'] = (100.0 * df_control['con_valor'] / df_control['total_registros']).round(2)
df_control['semaforo'] = np.where(df_control['sin_valor'] == 0, 'COMPLETO', 'ALERTA')

print(f'{len(df)} combinaciones fuente + cuenta + libro + mes '
      f'({len(df) - len(df_control)} solo con RAMO_PROD nulo — entran únicamente a Validez)')
print(f"Alertas de completitud: {(df_control['semaforo'] == 'ALERTA').sum()}")
df_control.head(10)

240 combinaciones fuente + cuenta + libro + mes (0 solo con RAMO_PROD nulo — entran únicamente a Validez)
Alertas de completitud: 0


,fuente,cuenta,libro,periodo_contable_analisis,total_registros,con_valor,sin_valor,ceros,ramo_nulo,valor_total,serie,pct_completitud,semaforo
0,DIRECTA_IAXIS,410310,AA,202601,46549,46549,0,0,0,-2.741921e+09,DIRECTA_IAXIS · 410310 · AA,100.0,COMPLETO
1,DIRECTA_IAXIS,410310,AA,202604,14829,14829,0,0,0,-1.116360e+09,DIRECTA_IAXIS · 410310 · AA,100.0,COMPLETO
2,DIRECTA_IAXIS,510305,AA,202603,481716,481716,0,0,0,2.269444e+09,DIRECTA_IAXIS · 510305 · AA,100.0,COMPLETO
3,DIRECTA_IAXIS,510310,AA,202511,69429,69429,0,0,0,-4.547701e+07,DIRECTA_IAXIS · 510310 · AA,100.0,COMPLETO
4,DIRECTA_IAXIS,510315,AL,202601,3956,3956,0,0,0,1.605999e+09,DIRECTA_IAXIS · 510315 · AL,100.0,COMPLETO
5,DIRECTA_IAXIS,410305,AL,202605,7286000,7286000,0,0,0,-3.640811e+10,DIRECTA_IAXIS · 410305 · AL,100.0,COMPLETO
6,DIRECTA_IAXIS,510305,AA,202508,308879,308879,0,0,0,-8.465878e+08,DIRECTA_IAXIS · 510305 · AA,100.0,COMPLETO
7,DIRECTA_IAXIS,510305,AA,202511,508939,508939,0,0,0,4.834568e+08,DIRECTA_IAXIS · 510305 · AA,100.0,COMPLETO
8,DIRECTA_IAXIS,410305,AL,202602,9792933,9792933,0,0,0,-4.977891e+10,DIRECTA_IAXIS · 410305 · AL,100.0,COMPLETO
9,DIRECTA_IAXIS,410310,AA,202509,5392,5392,0,0,0,-4.299893e+08,DIRECTA_IAXIS · 410310 · AA,100.0,COMPLETO


### 2.1 Validez — ¿la fila está bien formada?

Tres reglas de forma, todas calculadas sobre lo que ya trajo la consulta (sin volver a la base):

1. **`RAMO_PROD` presente** (solo cedidas): el control canónico descarta esas filas en silencio —
   si un mes aparecen muchas, algo cambió en la carga y nadie lo estaría viendo.
2. **Periodo bien formado**: el mes del `AAAAMM` debe estar entre 01 y 12.
3. **Libro no vacío**: un libro en blanco haría que la fila se agrupe mal en contabilidad.

El % de validez de una combinación es la proporción de filas que pasan las tres reglas.

In [9]:
mes = df['periodo_contable_analisis'].astype(int) % 100
df['periodo_invalido'] = ~mes.between(1, 12)
df['libro_vacio'] = df['libro'].eq('')

# En cedidas las filas con ramo nulo están FUERA de total_registros (el conteo canónico);
# en DIRECTA vienen incluidas. El universo de validez las suma donde haga falta.
es_cedida = df['fuente'].isin(['CEDIDAS_AS400', 'CEDIDAS_IAXIS'])
df['universo_validez'] = df['total_registros'] + np.where(es_cedida, df['ramo_nulo'], 0)

# Si el periodo o el libro de la combinación están mal, TODAS sus filas son inválidas;
# si no, las inválidas son las que no tienen ramo.
df['invalidas'] = np.where(df['periodo_invalido'] | df['libro_vacio'],
                           df['universo_validez'], df['ramo_nulo'])
df['pct_validez'] = (100.0 * (1 - df['invalidas'] / df['universo_validez'].replace(0, np.nan))).round(2)

con_invalidas = df[df['invalidas'] > 0]
print(f'{len(con_invalidas)} combinaciones con filas inválidas '
      f'({int(con_invalidas["invalidas"].sum())} filas en total)')
con_invalidas[['serie', 'periodo_contable_analisis', 'ramo_nulo',
               'invalidas', 'universo_validez', 'pct_validez']].head(10)

0 combinaciones con filas inválidas (0 filas en total)


,serie,periodo_contable_analisis,ramo_nulo,invalidas,universo_validez,pct_validez


### 2.2 Variación (Estabilidad) — ¿se movió más de lo normal?

Hasta acá revisamos la **forma** del dato. Ahora revisamos su **comportamiento**: esta es la
"Fase 2" (detección de valores atípicos) de `Hallazgos y Estado del Proyecto.md` — una primera
propuesta que hay que validar con el equipo.

Se revisan dos cosas por combinación fuente · cuenta · libro:

- **Variación de valor**: ¿la plata total de la reserva cambió mucho de un mes a otro?
- **Variación de volumen**: ¿la cantidad de filas cambió mucho? A veces el problema no es el
  valor sino que "desaparecieron" o "aparecieron" registros de la nada.

Una combinación se marca **atípica** si pasa cualquiera de estas dos pruebas:

1. Se aleja más de **2 desviaciones estándar** de su propio promedio en la ventana (z-score:
   qué tan raro es el valor comparado con lo que esa cuenta suele tener).
2. Cambió más de **20 %** respecto al mes inmediatamente anterior.

> Ya no hace falta otra consulta al DWH: `valor_total` vino en la misma pasada de la sección 2.

Los umbrales (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION`) están al inicio del código —
súbelos si el tablero avisa demasiado, o bájalos si avisa muy poco.

In [10]:
# UMBRALES: primera propuesta, hay que validarlos con el equipo.
UMBRAL_Z_VARIACION = 2.0     # desviaciones estándar respecto al promedio de la propia serie
UMBRAL_PCT_VARIACION = 0.20  # 20% de cambio respecto al mes inmediatamente anterior


def marcar_variacion_atipica(pivot, periodos,
                             umbral_z=UMBRAL_Z_VARIACION,
                             umbral_pct=UMBRAL_PCT_VARIACION,
                             nombre_id='serie'):
    '''
    Recibe una tabla (serie x periodo) y devuelve, en formato largo, una fila por
    serie+periodo con: el valor, su z-score respecto al promedio de esa misma serie
    en la ventana, la variación % contra el mes anterior, y si se marca atípica.
    '''
    promedio = pivot.mean(axis=1)
    desviacion = pivot.std(axis=1)
    filas = []
    for idx in pivot.index:
        valores = pivot.loc[idx]
        for i, periodo in enumerate(periodos):
            valor = valores.get(periodo)
            if pd.isna(valor):
                continue
            d = desviacion[idx]
            z = (valor - promedio[idx]) / d if d and not pd.isna(d) else float('nan')
            anterior = valores.get(periodos[i - 1]) if i > 0 else None
            if anterior is None or pd.isna(anterior):
                pct = float('nan')
            elif anterior == 0:
                pct = float('inf') if valor != 0 else 0.0
            else:
                pct = (valor - anterior) / abs(anterior)
            atipica = ((not pd.isna(z) and abs(z) > umbral_z)
                       or (not pd.isna(pct) and abs(pct) > umbral_pct))
            filas.append({
                nombre_id: idx,
                'periodo_contable_analisis': periodo,
                'valor': valor,
                'z_score': z,
                'pct_variacion': pct,
                'variacion_atipica': bool(atipica),
            })
    return pd.DataFrame(filas)


piv_valor = (df_control.pivot_table(index='serie', columns='periodo_contable_analisis',
                                    values='valor_total', aggfunc='sum')
             .reindex(columns=PERIODOS))
piv_vol = (df_control.pivot_table(index='serie', columns='periodo_contable_analisis',
                                  values='total_registros', aggfunc='sum')
           .reindex(columns=PERIODOS))

variacion_valor = marcar_variacion_atipica(piv_valor, PERIODOS)
variacion_vol = marcar_variacion_atipica(piv_vol, PERIODOS)

alertas_valor = (variacion_valor[variacion_valor['variacion_atipica']]
                 .sort_values('periodo_contable_analisis', ascending=False)
                 .reset_index(drop=True))
alertas_volumen = (variacion_vol[variacion_vol['variacion_atipica']]
                   .sort_values('periodo_contable_analisis', ascending=False)
                   .reset_index(drop=True))

print(f'{len(alertas_valor)} variaciones atípicas de VALOR en la ventana')
print(f'{len(alertas_volumen)} variaciones atípicas de VOLUMEN (cantidad de registros)')
alertas_valor.head(10)

153 variaciones atípicas de VALOR en la ventana
83 variaciones atípicas de VOLUMEN (cantidad de registros)


,serie,periodo_contable_analisis,valor,z_score,pct_variacion,variacion_atipica
0,CEDIDAS_AS400 · 510305 · AA,202606,8.083954e+08,1.409595,3.575248,True
1,CEDIDAS_IAXIS · 510305 · AL,202606,4.441608e+08,0.373344,-0.666169,True
2,DIRECTA_IAXIS · 510310 · AA,202606,2.259010e+09,1.365615,0.593122,True
3,CEDIDAS_TERREMOTO · 510305 · AL,202606,-5.958562e+08,0.191155,0.868762,True
4,CEDIDAS_TERREMOTO · 510305 · AA,202606,-4.896680e+08,-1.120584,-3.142477,True
5,CEDIDAS_TERREMOTO · 410305 · AL,202606,1.135412e+09,0.244918,-0.290825,True
6,CEDIDAS_TERREMOTO · 410305 · AA,202606,-5.096016e+06,0.037570,-1.002011,True
7,CEDIDAS_TERREMOTO · 263005 · AL,202606,-5.395555e+08,-0.507411,-1.183569,True
8,CEDIDAS_TERREMOTO · 263005 · AA,202606,4.947640e+08,0.286739,1.179068,True
9,DIRECTA_IAXIS · 410310 · AA,202606,-7.624850e+08,0.182717,0.231039,True


### 2.3 Estabilidad combinada (valor + volumen)

Para el tablero, las dos señales se combinan en un solo indicador, **Variación (Estabilidad)**:
una combinación se considera inestable si **cualquiera** de las dos (valor o volumen) se sale de
lo normal — así no hay que revisar dos pestañas para lo mismo. El puntaje va de 100 (sin cambios
atípicos) hacia 0, y está calibrado para caer justo en 80 (el límite de ALERTA) cuando se cruza
cualquiera de los dos umbrales de la sección 2.2.

In [11]:
# K_PCT y K_Z convierten z-score / % de variación en un puntaje 0-100 ("estabilidad"),
# calibrados para que el puntaje caiga exactamente a 80 (límite de ALERTA) cuando se
# cruza cualquiera de los dos umbrales de la sección 2.2.
K_PCT = 20.0 / UMBRAL_PCT_VARIACION
K_Z = 20.0 / UMBRAL_Z_VARIACION


def estabilidad_pct(pct_variacion, z_score):
    '''100 = sin cambios atípicos; baja hacia 0 mientras más se aleje de lo normal.'''
    base = 100.0
    if not pd.isna(pct_variacion):
        if pct_variacion in (float('inf'), float('-inf')):
            return 0.0
        base = min(base, 100.0 - min(100.0, K_PCT * abs(pct_variacion)))
    if not pd.isna(z_score):
        base = min(base, 100.0 - min(100.0, K_Z * abs(z_score)))
    return max(0.0, round(base, 2))


combinado = variacion_valor.merge(
    variacion_vol, on=['serie', 'periodo_contable_analisis'], how='outer',
    suffixes=('_valor', '_volumen'),
)
combinado = combinado.merge(
    df_control[['serie', 'periodo_contable_analisis', 'total_registros']].drop_duplicates(),
    on=['serie', 'periodo_contable_analisis'], how='left',
)

combinado['estabilidad'] = combinado.apply(
    lambda r: min(
        estabilidad_pct(r['pct_variacion_valor'], r['z_score_valor']),
        estabilidad_pct(r['pct_variacion_volumen'], r['z_score_volumen']),
    ),
    axis=1,
)
combinado['atipica'] = (
    combinado['variacion_atipica_valor'].fillna(False)
    | combinado['variacion_atipica_volumen'].fillna(False)
)
combinado['cantidad_mala'] = combinado.apply(
    lambda r: int(r['total_registros']) if r['atipica'] and not pd.isna(r['total_registros']) else 0,
    axis=1,
)

print(f'{len(combinado)} combinaciones con indicador de variación combinado')
combinado[['serie', 'periodo_contable_analisis', 'estabilidad', 'atipica', 'cantidad_mala']].head(10)

240 combinaciones con indicador de variación combinado


,serie,periodo_contable_analisis,estabilidad,atipica,cantidad_mala
0,CEDIDAS_AS400 · 410305 · AA,202507,90.07,False,0
1,CEDIDAS_AS400 · 410305 · AA,202508,93.06,False,0
2,CEDIDAS_AS400 · 410305 · AA,202509,90.07,False,0
3,CEDIDAS_AS400 · 410305 · AA,202510,90.89,False,0
4,CEDIDAS_AS400 · 410305 · AA,202511,87.32,False,0
5,CEDIDAS_AS400 · 410305 · AA,202512,84.64,False,0
6,CEDIDAS_AS400 · 410305 · AA,202601,52.51,True,4407
7,CEDIDAS_AS400 · 410305 · AA,202602,86.47,False,0
8,CEDIDAS_AS400 · 410305 · AA,202603,88.71,False,0
9,CEDIDAS_AS400 · 410305 · AA,202604,89.64,False,0


### 2.4 Disponibilidad, frescura y valores en cero

Tres señales de **proceso de carga** (informativas — no entran al DQ Score):

- **Huecos de carga**: una combinación cuenta/libro sin **ninguna** fila en un mes. Distinto de una
  alerta de completitud (ahí llegaron filas pero con el valor vacío); un hueco casi siempre significa
  que esa carga no corrió.
- **Frescura**: ¿las 4 tablas fuente tienen datos en cada mes? Una fuente rezagada baja este número
  aunque las demás estén al día.
- **Valores en cero**: filas con `VALOR_RESERVA_CONTABLE` presente pero exactamente en 0. Un cero
  puede ser legítimo (reserva liberada), pero un salto brusco en la cantidad de ceros suele anunciar
  un problema de carga — por eso se vigila la tendencia, sin marcarlo como error.

In [12]:
# --- Huecos de carga: combinación sin ninguna fila en un mes de la ventana.
huecos = (piv_vol.reset_index()
          .melt(id_vars='serie', var_name='periodo_contable_analisis', value_name='total_registros'))
huecos = (huecos[huecos['total_registros'].isna()]
          .drop(columns='total_registros')
          .sort_values(['serie', 'periodo_contable_analisis'])
          .reset_index(drop=True))

# --- Frescura: % de las 4 fuentes con al menos una fila por periodo.
fuentes_por_periodo = df_control.groupby('periodo_contable_analisis')['fuente'].unique()
frescura = {p: sorted(set(FUENTES) - set(fuentes_por_periodo.get(p, [])))
            for p in PERIODOS}
rezagadas_ultimo = frescura.get(ULTIMO_PERIODO, sorted(FUENTES))

# --- Valores en cero: agregado por periodo.
g_ceros = df_control.groupby('periodo_contable_analisis')[['ceros', 'total_registros']].sum()

print(f'{len(huecos)} huecos de carga en la ventana')
if rezagadas_ultimo:
    print(f'⚠️  Fuentes SIN datos en {ULTIMO_PERIODO}: {", ".join(rezagadas_ultimo)}')
else:
    print(f'Las {len(FUENTES)} fuentes tienen datos en {ULTIMO_PERIODO}.')
print('Filas en cero por periodo (últimos 3):')
print((100.0 * g_ceros['ceros'] / g_ceros['total_registros']).round(2).tail(3).to_string())

0 huecos de carga en la ventana
Las 4 fuentes tienen datos en 202606.
Filas en cero por periodo (últimos 3):
periodo_contable_analisis
202604    0.32
202605    0.43
202606    0.30


## 3) Consolidar el cubo en formato largo

Se unen todos los indicadores en un solo DataFrame con el mismo grano del cubo del tablero de
Primas (`periodo_contable` + `tipo_indicador` + `nombre_campo`), se agregan las filas de total
por periodo, y se guarda una copia en CSV (`reports/cubo_completitud_reservas.csv`) para poder
consultar la historia sin abrir el tablero.

In [13]:
COLUMNAS_CUBO = ['periodo_contable', 'tipo_indicador', 'nombre_campo',
                 'total_registros', 'cantidad_mala', 'porcentaje', 'es_total']

filas = []

# --- Completitud: detalle por combinación + total ponderado por registros.
for _, r in df_control.iterrows():
    filas.append({'periodo_contable': int(r['periodo_contable_analisis']),
                  'tipo_indicador': 'completitud', 'nombre_campo': r['serie'],
                  'total_registros': int(r['total_registros']),
                  'cantidad_mala': int(r['sin_valor']),
                  'porcentaje': float(r['pct_completitud']), 'es_total': False})
for p in PERIODOS:
    dp = df_control[df_control['periodo_contable_analisis'] == p]
    if dp.empty:
        continue
    tot, sin = int(dp['total_registros'].sum()), int(dp['sin_valor'].sum())
    filas.append({'periodo_contable': int(p), 'tipo_indicador': 'completitud',
                  'nombre_campo': 'TOTAL_PERIODO', 'total_registros': tot,
                  'cantidad_mala': sin,
                  'porcentaje': round(100.0 * (tot - sin) / tot, 2) if tot else 0.0,
                  'es_total': True})

# --- Validez: detalle + total ponderado (universo = filas evaluadas, incl. ramo nulo).
dv = df.dropna(subset=['pct_validez'])
for _, r in dv.iterrows():
    filas.append({'periodo_contable': int(r['periodo_contable_analisis']),
                  'tipo_indicador': 'validez', 'nombre_campo': r['serie'],
                  'total_registros': int(r['universo_validez']),
                  'cantidad_mala': int(r['invalidas']),
                  'porcentaje': float(r['pct_validez']), 'es_total': False})
for p in PERIODOS:
    dp = dv[dv['periodo_contable_analisis'] == p]
    if dp.empty:
        continue
    uni, inv = int(dp['universo_validez'].sum()), int(dp['invalidas'].sum())
    filas.append({'periodo_contable': int(p), 'tipo_indicador': 'validez',
                  'nombre_campo': 'TOTAL_PERIODO', 'total_registros': uni,
                  'cantidad_mala': inv,
                  'porcentaje': round(100.0 * (uni - inv) / uni, 2) if uni else 0.0,
                  'es_total': True})

# --- Variación (estabilidad combinada): detalle + total (promedio simple).
dc = combinado.dropna(subset=['estabilidad'])
for _, r in dc.iterrows():
    filas.append({'periodo_contable': int(r['periodo_contable_analisis']),
                  'tipo_indicador': 'variacion', 'nombre_campo': r['serie'],
                  'total_registros': int(r['total_registros']) if not pd.isna(r['total_registros']) else 0,
                  'cantidad_mala': int(r['cantidad_mala']),
                  'porcentaje': float(r['estabilidad']), 'es_total': False})
for p in PERIODOS:
    dp = dc[dc['periodo_contable_analisis'] == p]
    if dp.empty:
        continue
    filas.append({'periodo_contable': int(p), 'tipo_indicador': 'variacion',
                  'nombre_campo': 'TOTAL_PERIODO',
                  'total_registros': int(dp['total_registros'].fillna(0).sum()),
                  'cantidad_mala': int(dp['cantidad_mala'].sum()),
                  'porcentaje': round(float(dp['estabilidad'].mean()), 2), 'es_total': True})

# --- Disponibilidad de carga / Frescura / Valores en cero: una fila por periodo.
series_totales = df_control['serie'].nunique()
for p in PERIODOS:
    dp = df_control[df_control['periodo_contable_analisis'] == p]
    presentes = dp['serie'].nunique()
    huecos_p = int((huecos['periodo_contable_analisis'] == p).sum())
    filas.append({'periodo_contable': int(p), 'tipo_indicador': 'disponibilidad',
                  'nombre_campo': 'disponibilidad_carga', 'total_registros': int(series_totales),
                  'cantidad_mala': huecos_p,
                  'porcentaje': round(100.0 * presentes / series_totales, 2) if series_totales else 0.0,
                  'es_total': True})
    faltantes = frescura.get(p, [])
    filas.append({'periodo_contable': int(p), 'tipo_indicador': 'disponibilidad',
                  'nombre_campo': 'frescura_carga', 'total_registros': len(FUENTES),
                  'cantidad_mala': len(faltantes),
                  'porcentaje': round(100.0 * (len(FUENTES) - len(faltantes)) / len(FUENTES), 2),
                  'es_total': True})
    if p in g_ceros.index:
        tot, cer = int(g_ceros.loc[p, 'total_registros']), int(g_ceros.loc[p, 'ceros'])
        filas.append({'periodo_contable': int(p), 'tipo_indicador': 'disponibilidad',
                      'nombre_campo': 'valores_en_cero', 'total_registros': tot,
                      'cantidad_mala': cer,
                      'porcentaje': round(100.0 * (tot - cer) / tot, 2) if tot else 0.0,
                      'es_total': True})

df_cubo = (pd.DataFrame(filas)[COLUMNAS_CUBO]
           .sort_values(['periodo_contable', 'tipo_indicador', 'es_total', 'nombre_campo'])
           .reset_index(drop=True))

CSV_PATH = Path('..') / 'reports' / 'cubo_completitud_reservas.csv'
CSV_PATH.parent.mkdir(exist_ok=True)
df_cubo.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f'CSV guardado en {CSV_PATH.resolve()} ({len(df_cubo)} filas).')

# Vista rápida: porcentaje total por indicador y periodo.
df_cubo[df_cubo['es_total']].pivot_table(
    index=['tipo_indicador', 'nombre_campo'], columns='periodo_contable', values='porcentaje')

CSV guardado en C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\data-completeness-validator\reports\cubo_completitud_reservas.csv (792 filas).


periodo_contable                     202507  202508  202509  202510  202511  \
tipo_indicador nombre_campo                                                   
completitud    TOTAL_PERIODO         100.00  100.00  100.00  100.00  100.00   
disponibilidad disponibilidad_carga  100.00  100.00  100.00  100.00  100.00   
               frescura_carga        100.00  100.00  100.00  100.00  100.00   
               valores_en_cero        99.93   99.91   99.79   99.30   99.59   
validez        TOTAL_PERIODO         100.00  100.00  100.00  100.00  100.00   
variacion      TOTAL_PERIODO          91.83   46.05   44.24   38.23   49.08   

periodo_contable                     202512  202601  202602  202603  202604  \
tipo_indicador nombre_campo                                                   
completitud    TOTAL_PERIODO         100.00  100.00  100.00  100.00  100.00   
disponibilidad disponibilidad_carga  100.00  100.00  100.00  100.00  100.00   
               frescura_carga        100.00  100.00  100.00  100.00  100.00   
               valores_en_cero        99.61   99.52   99.88   99.78   99.68   
validez        TOTAL_PERIODO         100.00  100.00  100.00  100.00  100.00   
variacion      TOTAL_PERIODO          38.20    2.63   27.89   53.22   56.01   

periodo_contable                     202605  202606  
tipo_indicador nombre_campo                          
completitud    TOTAL_PERIODO         100.00  100.00  
disponibilidad disponibilidad_carga  100.00  100.00  
               frescura_carga        100.00  100.00  
               valores_en_cero        99.57   99.70  
validez        TOTAL_PERIODO         100.00  100.00  
variacion      TOTAL_PERIODO          48.80   49.62

## 4) Generar el tablero HTML de calidad

Se arma un archivo HTML **autocontenido** (sin dependencias externas, funciona sin internet) a
partir del cubo, con la misma plantilla corporativa de los otros tableros del equipo
(`notebooks/tablero_template.html`): aros de KPI, pestañas, tablas semáforo y modal de detalle
al hacer clic en un aro. Los datos se inyectan reemplazando el marcador `__DATA_JSON__`.

In [14]:
# Sin línea de target global: los umbrales por indicador ya marcan el estado.
TARGET = None

# Umbrales de estado por porcentaje (ajustables) — primera propuesta, validar con el equipo.
# Regla: >= BUENO -> BUENO; >= ALERTA -> ALERTA; >= GRAVE -> GRAVE; por debajo -> CRÍTICO.
UMBRALES = {
    # Completitud y Validez: casi no debería haber filas malas.
    'completitud':    {'BUENO': 99.5, 'ALERTA': 98, 'GRAVE': 95, 'CRÍTICO': 95},
    'validez':        {'BUENO': 99.5, 'ALERTA': 98, 'GRAVE': 95, 'CRÍTICO': 95},
    # Variación: calibrado para que 80 coincida justo con los umbrales de la sección 2.2.
    'variacion':      {'BUENO': 95,   'ALERTA': 80, 'GRAVE': 60, 'CRÍTICO': 60},
    # Disponibilidad de carga: pequeños huecos puntuales son tolerables.
    'disponibilidad': {'BUENO': 99,   'ALERTA': 95, 'GRAVE': 90, 'CRÍTICO': 90},
    # Frescura: con 4 fuentes, una sola rezagada ya deja el indicador en 75.
    'frescura':       {'BUENO': 99,   'ALERTA': 75, 'GRAVE': 50, 'CRÍTICO': 50},
    # Valores en cero: informativo — un % de ceros alto puede ser normal en reservas.
    'ceros':          {'BUENO': 90,   'ALERTA': 75, 'GRAVE': 50, 'CRÍTICO': 50},
    # Usado para el aro "Comportamiento general" (promedio de los indicadores).
    'default':        {'BUENO': 95,   'ALERTA': 90, 'GRAVE': 80, 'CRÍTICO': 80},
}

In [15]:
# Diccionario para el drill-down del tablero: qué es cada combinación fuente · cuenta · libro.
diccionario = {}
for serie in sorted(set(df['serie'])):
    fuente, cuenta, libro = serie.split(' · ')
    info = FUENTES[fuente]
    diccionario[serie] = (f'Cuenta {cuenta}, libro {libro or "(vacío)"} — '
                          f'{info["tabla"]} ({info["desc"]}).')

DATA = {
    'periodos': PERIODOS,
    'periodo_en_curso': PERIODO_EN_CURSO,
    'filas': filas,
    'target': TARGET,
    'umbrales': UMBRALES,
    'diccionario': diccionario,
    'generado': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'fuente': 'Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ, CEDIDAS_RESERVA_INTERFAZ, '
              'CEDIDAS_RESERVA_INTERFAZ_IAXIS, CEDIDAS_TERREMOTO_RESERVA_INTERFAZ '
              '(VALOR_RESERVA_CONTABLE)',
}

data_json = json.dumps(DATA, ensure_ascii=False)
plantilla = Path('tablero_template.html').read_text(encoding='utf-8')
html_doc = plantilla.replace('__DATA_JSON__', data_json)

ruta_html = Path('dashboard_completitud.html').resolve()
ruta_html.write_text(html_doc, encoding='utf-8')
print(f'Tablero guardado en: {ruta_html}')
webbrowser.open(ruta_html.as_uri())

Tablero guardado en: C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\data-completeness-validator\notebooks\dashboard_completitud.html

True

## 5) Dónde quedó el resultado

Dos archivos, ninguno necesita Python ni conexión al DWH para consultarse:

- **`notebooks/dashboard_completitud.html`** — tablero interactivo autocontenido (doble clic y se
  abre en el navegador). Cinco pestañas:
  - **Resumen ejecutivo**: aros de Completitud, Validez y Variación (Estabilidad) + DQ Score, y
    tarjetas informativas de Disponibilidad de carga, Frescura y Valores en cero.
  - **Detalle por combinación**: tabla fuente · cuenta · libro por indicador, de peor a mejor.
  - **Tendencia e historia**: evolución de la ventana y variación vs. el mes anterior.
  - **Análisis de alertas**: distribución por indicador y Pareto 80/20 de las combinaciones.
  - **Información**: qué mide cada indicador, umbrales y notas metodológicas.

  Haz clic en cualquier aro del resumen para ver en qué combinación falla (modal de drill-down).

- **`reports/cubo_completitud_reservas.csv`** — el cubo en formato largo, por si quieres cruzar la
  historia en Excel/Power BI sin correr el notebook.

Recordatorios:

- Cuentas excluidas por diseño: Incurrido, Salvamentos, Recobros, IVA AG — **+ 2 cuentas pendientes
  de confirmar con Andrey**.
- Todos los umbrales (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION` y los de estado en `UMBRALES`)
  son una primera propuesta — es la "Fase 2" de `Hallazgos y Estado del Proyecto.md`, pendiente de
  validar con el equipo.
- Si cambias el diseño visual, edita `notebooks/tablero_template.html` (no este notebook) — así el
  notebook se queda solo con la lógica de datos.

## Cerrar conexión

In [16]:
engine.dispose()
print('Conexión cerrada.')

Conexión cerrada.
